In [1]:
from pathlib import Path
import pandas as pd
import re

# Define paths
data_folder = Path("../data")
processed_folder = Path("../data/processed")
evidence_folder = Path("../outputs/evidence_tables")

company_file = data_folder / "company_master_list.csv"
keyword_mentions_file = processed_folder / "ai_keyword_mentions.csv"
ai_mentions_file = evidence_folder / "ai_transparency_mentions_detailed.csv"
company_ai_summary_file = evidence_folder / "company_ai_transparency_summary.csv"

# Load files
company_master = pd.read_csv(company_file, dtype={"company_id": str})
keyword_mentions = pd.read_csv(keyword_mentions_file, dtype={"company_id": str})
ai_mentions = pd.read_csv(ai_mentions_file, dtype={"company_id": str})
company_ai_summary = pd.read_csv(company_ai_summary_file, dtype={"company_id": str})

print("Company master rows:", len(company_master))
print("All keyword mentions:", len(keyword_mentions))
print("AI-transparency mentions:", len(ai_mentions))
print("Company AI summary rows:", len(company_ai_summary))

company_ai_summary.head()

Company master rows: 20
All keyword mentions: 13117
AI-transparency mentions: 1452
Company AI summary rows: 20


,company_id,company_name,total_ai_transparency_mentions,unique_pages_with_mentions,unique_keywords_found,keywords_found
0,001,Barclays plc,249,60,10,"AI, artificial intelligence, automation, bias,..."
1,004,NatWest Group plc,210,61,10,"AI, AI ethics, artificial intelligence, automa..."
2,019,London Stock Exchange Group plc,119,34,8,"AI, LLM, artificial intelligence, automation, ..."
3,016,Funding Circle Holdings plc,104,30,8,"AI, artificial intelligence, automation, bias,..."
4,009,Admiral Group plc,103,39,8,"AI, artificial intelligence, automation, bias,..."


In [2]:
# Scoring dimensions:
# Each dimension is scored from 0 to 2.
#
# 0 = no clear evidence
# 1 = limited or generic evidence
# 2 = clear and repeated evidence

def score_by_count(count, low_threshold=1, high_threshold=5):
    if count >= high_threshold:
        return 2
    elif count >= low_threshold:
        return 1
    else:
        return 0


def contains_any(text, terms):
    text = str(text).lower()
    return any(term.lower() in text for term in terms)


# Keyword groups for scoring

core_ai_keywords = [
    "AI",
    "artificial intelligence",
    "machine learning",
    "generative AI",
    "large language model",
    "LLM"
]

business_use_keywords = [
    "automation",
    "algorithm",
    "machine learning",
    "decision support",
    "automated decision-making",
    "data analytics",
    "predictive analytics"
]

risk_terms = [
    "risk",
    "model risk",
    "regulatory",
    "compliance",
    "privacy",
    "data protection",
    "cybersecurity",
    "security",
    "control"
]

governance_terms = [
    "governance",
    "accountability",
    "oversight",
    "board",
    "committee",
    "framework",
    "policy",
    "responsible AI"
]

human_oversight_keywords = [
    "human oversight",
    "human review",
    "human-in-the-loop",
    "automated decision-making",
    "decision support"
]

ethics_fairness_keywords = [
    "bias",
    "fairness",
    "discrimination",
    "ethics",
    "ethical AI",
    "AI ethics",
    "responsible AI"
]

In [3]:
scoring_rows = []

for _, company in company_master.iterrows():
    
    company_id = company["company_id"]
    company_name = company["company_name"]
    report_year = company["annual_report_year"]
    
    company_mentions = ai_mentions[
        ai_mentions["company_id"] == company_id
    ].copy()
    
    all_company_mentions = keyword_mentions[
        keyword_mentions["company_id"] == company_id
    ].copy()
    
    # Count core AI terms
    core_ai_count = company_mentions[
        company_mentions["keyword"].isin(core_ai_keywords)
    ].shape[0]
    
    # Business-use evidence
    business_use_count = company_mentions[
        company_mentions["keyword"].isin(business_use_keywords)
    ].shape[0]
    
    # Risk evidence based on context around AI-related mentions
    risk_context_count = company_mentions[
        company_mentions["context"].apply(
            lambda x: contains_any(x, risk_terms)
        )
    ].shape[0]
    
    # Governance evidence based on context around AI-related mentions
    governance_context_count = company_mentions[
        company_mentions["context"].apply(
            lambda x: contains_any(x, governance_terms)
        )
    ].shape[0]
    
    # Human oversight evidence
    human_oversight_count = company_mentions[
        company_mentions["keyword"].isin(human_oversight_keywords)
    ].shape[0]
    
    # Ethics, fairness and bias evidence
    ethics_fairness_count = company_mentions[
        company_mentions["keyword"].isin(ethics_fairness_keywords)
    ].shape[0]
    
    # Specificity proxy
    total_mentions = company_mentions.shape[0]
    unique_keywords = company_mentions["keyword"].nunique()
    unique_pages = company_mentions["page_number"].nunique()
    
    if total_mentions >= 50 and unique_keywords >= 7 and unique_pages >= 20:
        specificity_score = 2
    elif total_mentions >= 10 and unique_keywords >= 3:
        specificity_score = 1
    else:
        specificity_score = 0
    
    # Individual scores
    ai_mention_score = score_by_count(core_ai_count, 1, 5)
    business_use_score = score_by_count(business_use_count, 1, 5)
    risk_score = score_by_count(risk_context_count, 1, 5)
    governance_score = score_by_count(governance_context_count, 1, 5)
    human_oversight_score = score_by_count(human_oversight_count, 1, 3)
    ethics_fairness_score = score_by_count(ethics_fairness_count, 1, 5)
    
    total_score = (
        ai_mention_score
        + business_use_score
        + risk_score
        + governance_score
        + human_oversight_score
        + ethics_fairness_score
        + specificity_score
    )
    
    score_percentage = round((total_score / 14) * 100, 2)
    
    # Pick one supporting context extract
    if not company_mentions.empty:
        supporting_context = company_mentions.iloc[0]["context"]
    else:
        supporting_context = ""
    
    coder_notes = (
        "Initial automated score based on keyword frequency and context. "
        "Manual validation required before using as final dissertation score."
    )
    
    scoring_rows.append({
        "company_id": company_id,
        "company_name": company_name,
        "report_year": report_year,
        "ai_mention_score": ai_mention_score,
        "business_use_score": business_use_score,
        "risk_score": risk_score,
        "governance_score": governance_score,
        "human_oversight_score": human_oversight_score,
        "ethics_fairness_score": ethics_fairness_score,
        "specificity_score": specificity_score,
        "total_score": total_score,
        "score_percentage": score_percentage,
        "coder_notes": coder_notes,
        "supporting_quote_or_context": supporting_context
    })

transparency_scores = pd.DataFrame(scoring_rows)

transparency_scores = transparency_scores.sort_values(
    by=["total_score", "score_percentage"],
    ascending=False
)

transparency_scores

,company_id,company_name,report_year,ai_mention_score,business_use_score,risk_score,governance_score,human_oversight_score,ethics_fairness_score,specificity_score,total_score,score_percentage,coder_notes,supporting_quote_or_context
0,001,Barclays plc,2025,2,2,2,2,1,2,2,13,92.86,Initial automated score based on keyword frequ...,class customer experience; and delivering best...
3,004,NatWest Group plc,2025,2,2,2,2,1,2,2,13,92.86,Initial automated score based on keyword frequ...,"for example, privacy training to our data cust..."
1,002,HSBC Holdings plc,2025,2,2,2,2,0,2,2,12,85.71,Initial automated score based on keyword frequ...,"source of funding for us, and forms the Hongko..."
8,009,Admiral Group plc,2025,2,2,2,2,0,2,2,12,85.71,Initial automated score based on keyword frequ...,ese shifts by leveraging our strengths across ...
15,016,Funding Circle Holdings plc,2025,2,2,2,2,1,1,2,12,85.71,Initial automated score based on keyword frequ...,ats: the dual nature of AI environment. By mai...
18,019,London Stock Exchange Group plc,2025,2,2,2,2,0,2,2,12,85.71,Initial automated score based on keyword frequ...,n track record of delivering real value throug...
19,020,Experian plc,2025,2,2,2,2,0,2,2,12,85.71,Initial automated score based on keyword frequ...,"rating of 4.2/5, Our client Net Promotor Score..."
4,005,Standard Chartered plc,2025,2,1,2,2,0,2,2,11,78.57,Initial automated score based on keyword frequ...,"ty, sustainability challenges and Group’s stra..."
14,015,CAB Payments Holdings plc,2025,2,2,2,2,0,2,1,11,78.57,Initial automated score based on keyword frequ...,"authorities to ensure a compliant this, artifi..."
2,003,Lloyds Banking Group plc,2025,2,1,2,2,0,1,2,10,71.43,Initial automated score based on keyword frequ...,verview Competition remains intense with high ...


In [4]:
# Save to main data file
transparency_scores_file = data_folder / "transparency_scores.csv"

transparency_scores.to_csv(
    transparency_scores_file,
    index=False
)

# Save ranking copy to evidence folder
ranking_file = evidence_folder / "initial_transparency_score_ranking.csv"

transparency_scores.to_csv(
    ranking_file,
    index=False
)

print(f"Transparency scores saved to: {transparency_scores_file}")
print(f"Ranking table saved to: {ranking_file}")

transparency_scores[
    [
        "company_id",
        "company_name",
        "ai_mention_score",
        "business_use_score",
        "risk_score",
        "governance_score",
        "human_oversight_score",
        "ethics_fairness_score",
        "specificity_score",
        "total_score",
        "score_percentage"
    ]
]

Transparency scores saved to: ..\data\transparency_scores.csv
Ranking table saved to: ..\outputs\evidence_tables\initial_transparency_score_ranking.csv


,company_id,company_name,ai_mention_score,business_use_score,risk_score,governance_score,human_oversight_score,ethics_fairness_score,specificity_score,total_score,score_percentage
0,001,Barclays plc,2,2,2,2,1,2,2,13,92.86
3,004,NatWest Group plc,2,2,2,2,1,2,2,13,92.86
1,002,HSBC Holdings plc,2,2,2,2,0,2,2,12,85.71
8,009,Admiral Group plc,2,2,2,2,0,2,2,12,85.71
15,016,Funding Circle Holdings plc,2,2,2,2,1,1,2,12,85.71
18,019,London Stock Exchange Group plc,2,2,2,2,0,2,2,12,85.71
19,020,Experian plc,2,2,2,2,0,2,2,12,85.71
4,005,Standard Chartered plc,2,1,2,2,0,2,2,11,78.57
14,015,CAB Payments Holdings plc,2,2,2,2,0,2,1,11,78.57
2,003,Lloyds Banking Group plc,2,1,2,2,0,1,2,10,71.43


The first automated scoring is too generous. Many companies are getting 2 for most categories because annual reports mention broad terms like risk, governance and accountability very often. 

In [6]:
# Stricter scoring version
# This reduces over-scoring from broad annual-report terms.

def score_strict_count(count, low_threshold=2, high_threshold=10):
    if count >= high_threshold:
        return 2
    elif count >= low_threshold:
        return 1
    else:
        return 0


strict_scoring_rows = []

for _, company in company_master.iterrows():
    
    company_id = company["company_id"]
    company_name = company["company_name"]
    report_year = company["annual_report_year"]
    
    company_mentions = ai_mentions[
        ai_mentions["company_id"] == company_id
    ].copy()
    
    # Counts by keyword
    keyword_counts = (
        company_mentions["keyword"]
        .value_counts()
        .to_dict()
    )
    
    core_ai_count = company_mentions[
        company_mentions["keyword"].isin(
            [
                "artificial intelligence",
                "AI",
                "machine learning",
                "generative AI",
                "large language model",
                "LLM"
            ]
        )
    ].shape[0]
    
    business_use_count = company_mentions[
        company_mentions["keyword"].isin(
            [
                "automation",
                "algorithm",
                "machine learning",
                "automated decision-making",
                "decision support"
            ]
        )
    ].shape[0]
    
    ai_governance_count = company_mentions[
        company_mentions["keyword"].isin(
            [
                "responsible AI",
                "ethical AI",
                "AI ethics"
            ]
        )
    ].shape[0]
    
    human_oversight_count = company_mentions[
        company_mentions["keyword"].isin(
            [
                "human oversight",
                "human review",
                "human-in-the-loop",
                "automated decision-making"
            ]
        )
    ].shape[0]
    
    ethics_fairness_count = company_mentions[
        company_mentions["keyword"].isin(
            [
                "bias",
                "fairness",
                "discrimination",
                "ethical AI",
                "AI ethics",
                "responsible AI"
            ]
        )
    ].shape[0]
    
    # Risk score is stricter:
    # it only counts AI-related mentions where the context also contains risk-type language.
    risk_context_count = company_mentions[
        company_mentions["context"].apply(
            lambda x: contains_any(
                x,
                [
                    "risk",
                    "model risk",
                    "regulatory",
                    "compliance",
                    "privacy",
                    "data protection",
                    "cybersecurity",
                    "security",
                    "control"
                ]
            )
        )
    ].shape[0]
    
    total_mentions = company_mentions.shape[0]
    unique_keywords = company_mentions["keyword"].nunique()
    unique_pages = company_mentions["page_number"].nunique()
    
    # Scoring
    ai_mention_score = score_strict_count(core_ai_count, 2, 10)
    business_use_score = score_strict_count(business_use_count, 2, 10)
    risk_score = score_strict_count(risk_context_count, 3, 15)
    governance_score = score_strict_count(ai_governance_count, 1, 3)
    human_oversight_score = score_strict_count(human_oversight_count, 1, 3)
    ethics_fairness_score = score_strict_count(ethics_fairness_count, 2, 10)
    
    if total_mentions >= 75 and unique_keywords >= 8 and unique_pages >= 30:
        specificity_score = 2
    elif total_mentions >= 20 and unique_keywords >= 5 and unique_pages >= 10:
        specificity_score = 1
    else:
        specificity_score = 0
    
    total_score = (
        ai_mention_score
        + business_use_score
        + risk_score
        + governance_score
        + human_oversight_score
        + ethics_fairness_score
        + specificity_score
    )
    
    score_percentage = round((total_score / 14) * 100, 2)
    
    strict_scoring_rows.append({
        "company_id": company_id,
        "company_name": company_name,
        "report_year": report_year,
        "core_ai_count": core_ai_count,
        "business_use_count": business_use_count,
        "risk_context_count": risk_context_count,
        "ai_governance_count": ai_governance_count,
        "human_oversight_count": human_oversight_count,
        "ethics_fairness_count": ethics_fairness_count,
        "total_ai_transparency_mentions": total_mentions,
        "unique_keywords": unique_keywords,
        "unique_pages": unique_pages,
        "ai_mention_score": ai_mention_score,
        "business_use_score": business_use_score,
        "risk_score": risk_score,
        "governance_score": governance_score,
        "human_oversight_score": human_oversight_score,
        "ethics_fairness_score": ethics_fairness_score,
        "specificity_score": specificity_score,
        "total_score": total_score,
        "score_percentage": score_percentage,
        "coder_notes": "Stricter automated score. Manual validation still required before final dissertation use."
    })

strict_transparency_scores = pd.DataFrame(strict_scoring_rows)

strict_transparency_scores = strict_transparency_scores.sort_values(
    by=["total_score", "score_percentage", "total_ai_transparency_mentions"],
    ascending=False
)

strict_transparency_scores

,company_id,company_name,report_year,core_ai_count,business_use_count,risk_context_count,ai_governance_count,human_oversight_count,ethics_fairness_count,total_ai_transparency_mentions,...,ai_mention_score,business_use_score,risk_score,governance_score,human_oversight_score,ethics_fairness_score,specificity_score,total_score,score_percentage,coder_notes
3,004,NatWest Group plc,2025,183,18,86,5,1,11,210,...,2,2,2,2,1,2,2,13,92.86,Stricter automated score. Manual validation st...
0,001,Barclays plc,2025,224,14,132,2,1,17,249,...,2,2,2,1,1,2,2,12,85.71,Stricter automated score. Manual validation st...
8,009,Admiral Group plc,2025,87,8,39,7,0,13,103,...,2,1,2,2,0,2,2,11,78.57,Stricter automated score. Manual validation st...
15,016,Funding Circle Holdings plc,2025,96,5,53,0,1,4,104,...,2,1,2,0,1,1,2,9,64.29,Stricter automated score. Manual validation st...
1,002,HSBC Holdings plc,2025,89,8,44,2,0,9,102,...,2,1,2,1,0,1,2,9,64.29,Stricter automated score. Manual validation st...
4,005,Standard Chartered plc,2025,84,3,32,3,0,8,95,...,2,1,2,2,0,1,1,9,64.29,Stricter automated score. Manual validation st...
2,003,Lloyds Banking Group plc,2025,73,4,36,2,0,3,79,...,2,1,2,1,0,1,2,9,64.29,Stricter automated score. Manual validation st...
18,019,London Stock Exchange Group plc,2025,108,5,23,0,0,7,119,...,2,1,2,0,0,1,2,8,57.14,Stricter automated score. Manual validation st...
19,020,Experian plc,2025,60,9,25,0,0,7,74,...,2,1,2,0,0,1,1,7,50.00,Stricter automated score. Manual validation st...
5,006,Aviva plc,2025,84,6,15,0,0,1,91,...,2,1,2,0,0,0,1,6,42.86,Stricter automated score. Manual validation st...


In [7]:
# Cleaner view of strict transparency scoring results

strict_score_view = strict_transparency_scores[
    [
        "company_id",
        "company_name",
        "total_ai_transparency_mentions",
        "unique_keywords",
        "unique_pages",
        "ai_mention_score",
        "business_use_score",
        "risk_score",
        "governance_score",
        "human_oversight_score",
        "ethics_fairness_score",
        "specificity_score",
        "total_score",
        "score_percentage"
    ]
].reset_index(drop=True)

strict_score_view

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,ai_mention_score,business_use_score,risk_score,governance_score,human_oversight_score,ethics_fairness_score,specificity_score,total_score,score_percentage
0,004,NatWest Group plc,210,10,61,2,2,2,2,1,2,2,13,92.86
1,001,Barclays plc,249,10,60,2,2,2,1,1,2,2,12,85.71
2,009,Admiral Group plc,103,8,39,2,1,2,2,0,2,2,11,78.57
3,016,Funding Circle Holdings plc,104,8,30,2,1,2,0,1,1,2,9,64.29
4,002,HSBC Holdings plc,102,10,39,2,1,2,1,0,1,2,9,64.29
5,005,Standard Chartered plc,95,7,33,2,1,2,2,0,1,1,9,64.29
6,003,Lloyds Banking Group plc,79,8,35,2,1,2,1,0,1,2,9,64.29
7,019,London Stock Exchange Group plc,119,8,34,2,1,2,0,0,1,2,8,57.14
8,020,Experian plc,74,7,35,2,1,2,0,0,1,1,7,50.00
9,006,Aviva plc,91,5,33,2,1,2,0,0,0,1,6,42.86


In [8]:
strict_score_ranking = strict_transparency_scores[
    [
        "company_id",
        "company_name",
        "total_ai_transparency_mentions",
        "unique_keywords",
        "unique_pages",
        "total_score",
        "score_percentage"
    ]
].reset_index(drop=True)

strict_score_ranking

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage
0,004,NatWest Group plc,210,10,61,13,92.86
1,001,Barclays plc,249,10,60,12,85.71
2,009,Admiral Group plc,103,8,39,11,78.57
3,016,Funding Circle Holdings plc,104,8,30,9,64.29
4,002,HSBC Holdings plc,102,10,39,9,64.29
5,005,Standard Chartered plc,95,7,33,9,64.29
6,003,Lloyds Banking Group plc,79,8,35,9,64.29
7,019,London Stock Exchange Group plc,119,8,34,8,57.14
8,020,Experian plc,74,7,35,7,50.00
9,006,Aviva plc,91,5,33,6,42.86


In [9]:
# Save stricter automated scoring as the main current transparency score

strict_transparency_scores.to_csv(
    data_folder / "transparency_scores.csv",
    index=False
)

strict_transparency_scores.to_csv(
    evidence_folder / "strict_transparency_score_ranking.csv",
    index=False
)

strict_score_ranking.to_csv(
    evidence_folder / "strict_transparency_score_ranking_simple.csv",
    index=False
)

print("Saved stricter transparency scores to:")
print(data_folder / "transparency_scores.csv")
print(evidence_folder / "strict_transparency_score_ranking.csv")
print(evidence_folder / "strict_transparency_score_ranking_simple.csv")

Saved stricter transparency scores to:
..\data\transparency_scores.csv
..\outputs\evidence_tables\strict_transparency_score_ranking.csv
..\outputs\evidence_tables\strict_transparency_score_ranking_simple.csv


In [10]:
#Creating a munual valiidation file
manual_validation = strict_score_ranking.copy()

manual_validation["manual_adjusted_score"] = ""
manual_validation["manual_adjusted_percentage"] = ""
manual_validation["reason_for_adjustment"] = ""
manual_validation["important_pages"] = ""
manual_validation["important_extracts"] = ""
manual_validation["review_status"] = "Pending manual review"

manual_validation.to_csv(
    data_folder / "manual_review_notes.csv",
    index=False
)

manual_validation

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
0,004,NatWest Group plc,210,10,61,13,92.86,,,,,,Pending manual review
1,001,Barclays plc,249,10,60,12,85.71,,,,,,Pending manual review
2,009,Admiral Group plc,103,8,39,11,78.57,,,,,,Pending manual review
3,016,Funding Circle Holdings plc,104,8,30,9,64.29,,,,,,Pending manual review
4,002,HSBC Holdings plc,102,10,39,9,64.29,,,,,,Pending manual review
5,005,Standard Chartered plc,95,7,33,9,64.29,,,,,,Pending manual review
6,003,Lloyds Banking Group plc,79,8,35,9,64.29,,,,,,Pending manual review
7,019,London Stock Exchange Group plc,119,8,34,8,57.14,,,,,,Pending manual review
8,020,Experian plc,74,7,35,7,50.00,,,,,,Pending manual review
9,006,Aviva plc,91,5,33,6,42.86,,,,,,Pending manual review
